this code simulates a LED diode as source,
light diffract at primary lens at 220μm from diode, (through LED medium of epoxy)
then diffract at a secondary lens (through glass medium)

In [ ]:
config_path = "RS_results/LED_double_collimator/config.yaml"
import numpy as np; import matplotlib.pyplot as plt # type: ignore
from datetime import datetime
import src.RS as RS
import src.math_tool as mt
import src.img_tool as it
import src.load_config as lc
import src.utils as utils
from tqdm import tqdm
import gc
import src.ASM as ASM
%matplotlib widget
%load_ext autoreload
%autoreload 2
c, π = lc.load_RS(config_path), np.pi
Axs2D, Ays2D = np.meshgrid(c.A.xs, c.A.ys, indexing='ij')
Nθs = 64
θ0s = np.linspace(0, 2*π, Nθs, endpoint=False, dtype=np.float32)
n_G = 1.5 # refractive index in glass
n_L = 1.5 # refractive index in LED casing (epoxy)

1. Design of primary lens

In [ ]:
Axs2D, Ays2D = np.meshgrid(c.A.xs, c.A.ys, indexing='ij')
Axy = np.zeros_like(Axs2D, dtype=complex)
# config the Aperture plane

#simulate wave from a point on LED diode propagate to pinhole, in medium of refractive index n_L (probably epoxy)
x_L_point = np.array([0])
y_L_point = np.array([0])
z_L_point = np.array([-220]) # N_point # location of electron hole recombination
θ_L_point = np.array([0])
magnitude_L_point = np.array([1])
# Rayleigh Sommerfeld summation
λ = c.λ/n_L
k = c.k * n_L
r = mt.dist(Axs2D[:, :, None], Ays2D[:, :, None], z=0, x_c = x_L_point, y_c=y_L_point, z_c = z_L_point) # Nx, Ny, N_point
E_center_0 = (-1j/λ * np.exp(k*1j*r)*z_L_point/r**2)@(np.exp(1j*θ_L_point)*magnitude_L_point) # Nx, Ny

In [ ]:
# target wave at focal 22μm (expected to shrink LED by 10x)
#simulate wave from a point on LED diode propagate to pinhole, in medium of refractive index n_L (probably epoxy)
x_L_point = np.array([0])
y_L_point = np.array([0])
z_L_point = np.array([-22]) # N_point # location of electron hole recombination
θ_L_point = np.array([0])
magnitude_L_point = np.array([1])
# Rayleigh Sommerfeld summation
n_L = 1.5
λ = c.λ/n_L
k = c.k * n_L
r = mt.dist(Axs2D[:, :, None], Ays2D[:, :, None], z=0, x_c = x_L_point, y_c=y_L_point, z_c = z_L_point) # Nx, Ny, N_point
E_center_target_0 = (-1j/λ * np.exp(k*1j*r)*z_L_point/r**2)@(np.exp(1j*θ_L_point)*magnitude_L_point) # Nx, Ny

In [ ]:
E_relative_0 = E_center_0 / E_center_target_0
θ_aperture = np.angle(E_relative_0)
cos_θ_aperture_θ0s = np.cos(θ_aperture[..., None]-θ0s) # Nx, Ny, Nθs
# print(cos_θ_aperture_θ0s.shape)
edge = it.block_edge_mask(c.A.Nx, c.A.Ny, int(300/2/c.A.Lx*c.A.Nx)) # Nx, Ny # circle mask of Nx x Ny pixel with pixel radius
# apply constraint of 100μm diameter
aperture_design = (cos_θ_aperture_θ0s > 0) * ( edge[:, :, None] == True ) # set edge to block
# find best possible match over range of reference phase
#                  phase_similarity (Nx, Ny, Nθs)
cosine_similarity = (cos_θ_aperture_θ0s * aperture_design * np.abs(E_center_0)[:, :, None]).sum(axis=(0, 1)) # Nθs (range of reference phase)
idx_max = cosine_similarity.argmax()
idx_min = cosine_similarity.argmin()
print(f'min similarity phase / max similarity phase\n{cosine_similarity[idx_min] / cosine_similarity[idx_max]}')
import gc
del cos_θ_aperture_θ0s
primary_lens = aperture_design[:, :, idx_max]
del aperture_design
gc.collect()

In [ ]:
plt.imshow(primary_lens.T);plt.gca().invert_yaxis()
plt.show()
name = 'LED_-220μm_2_-22μm_λ620nm_300_primary'
it.array_2_img(primary_lens, f'{c.path.results}/{name}.bmp')

2. Simulation test light from a point on LED diode to propagate to desigend primary lens (E_test)

In [ ]:
x_L_point = np.array([30])
y_L_point = np.array([0])
z_L_point = np.array([-220]) # N_point # location of electron hole recombination
θ_L_point = np.array([0])
magnitude_L_point = np.array([1])
# Rayleigh Sommerfeld summation
n_L = 1.5
λ = c.λ/n_L
k = c.k * n_L
r = mt.dist(Axs2D[:, :, None], Ays2D[:, :, None], z=0, x_c = x_L_point, y_c=y_L_point, z_c = z_L_point) # Nx, Ny, N_point
E_test_0 = (-1j/λ * np.exp(k*1j*r)*z_L_point/r**2)@(np.exp(1j*θ_L_point)*magnitude_L_point) # Nx, Ny
E_test_0 *= primary_lens

3. Simulate test light from primary lens to secondary lens

In [ ]:
# simulate light propagate from primary lens
import src.ASM as ASM
c.sim.z_start = 0
c.sim.z_end = 1100
c.sim.Nz = 110
c.sim.zs = np.linspace(c.sim.z_start, c.sim.z_end, c.sim.Nz, endpoint=False)
c.sim.Δz = c.sim.zs[1] - c.sim.zs[0]
pad_value = 300
mul = 2*pad_value//len(E_center_0)+1
ASM.ASM_3D_batch_E_Multi_Process(A_xy=it.img_pad(E_test_0, (pad_value,pad_value,pad_value,pad_value), pad_val=0), Lx = c.A.Lx*mul, Ly=c.A.Ly*mul, zs=c.sim.zs,\
                                  λ=c.λ/n_G, path=c.path.E,\
                                  batch_size=4, xy_range_idx=(pad_value, -pad_value, pad_value, -pad_value), num_process=6)
time = datetime.now()
print(f'last run: {time.isoformat(timespec="minutes")}')

In [ ]:
# extract results
E_test_0_1100_110 = ASM.batch_E_extract(c.A.Nx, c.A.Ny, len(c.sim.zs), c.path.E, batch_size=4)
time = datetime.now()
print(f'last run: {time.isoformat(timespec="minutes")}')
E_test_1000 = E_test_0_1100_110[100]

In [ ]:
# E2 =np.abs(E_test_0_1100_110)**2
# # x, y plot at different zs
# E2_max = E2.max(axis=(0, 1)) # Nz # find peak intensity
# s = 0
# order = np.flip(np.argsort(E2_max[s:])) + s
# focal_z_idx = order[0]
# layout = [[0, 1, 2], [3,3,3]]
# plot_z_idxs = [0, 2, focal_z_idx]                                                              # <-- edit plotted z-location here
# E2_plot = [E2[:, :, plot_z_idxs[0]], E2[:, :, plot_z_idxs[1]], E2[:, :, plot_z_idxs[2]]]
# plt.clf()
# fig, ax = plt.subplot_mosaic(layout, figsize=(18,12))
# fig.suptitle(f'diffraction pattern λ={c.λ*1e3}nm\naxes units: μm', fontsize=20)
# fig.tight_layout(pad=1.5)
# for i in range(len(plot_z_idxs)):
#     tmp = ax[i].imshow(E2_plot[i].T, aspect='equal', cmap='plasma')
#     ax[i].invert_yaxis()
#     ax[i].set_xticks(c.sim.x_tick_idxs, c.sim.xs[c.sim.x_tick_idxs])
#     ax[i].set_yticks(c.sim.y_tick_idxs, c.sim.ys[c.sim.y_tick_idxs])
#     ax[i].set_title(f'Z={c.sim.zs[plot_z_idxs[i]]}μm')
#     fig.colorbar(tmp, ax=[ax[i]], location='top')
# y_idx = c.sim.Ny//2
# ax[3].set_title(f'x-z plane (y={c.sim.ys[y_idx]}μm)')
# ax[3].invert_yaxis()
# tmp = ax[3].imshow(E2[:, y_idx, :], aspect='auto', cmap='plasma')
# #ax[3].set_xticks(c.z_tick_idxs, c.sim.zs[c.z_tick_idxs]);
# ax[3].set_yticks(c.sim.x_tick_idxs, c.sim.xs[c.sim.x_tick_idxs])
# ax[3].axvline(x=plot_z_idxs[1], color='r'); ax[3].axvline(x=plot_z_idxs[2], color='r')
# ax[3].set_xlabel('z axis'); ax[3].set_ylabel('x axis')
# fig.colorbar(tmp, ax=[ax[3]], location='right', fraction=0.05, aspect=50)
# #plt.tight_layout()
# plt.savefig(f'{c.path.results}/diffraction.png', bbox_inches='tight')
# plt.show()
# s, e = 0, 10
# print(f'{s}-th to {e}-th highest intensity\n{E2_max[order][s:e]}\nat\n{c.sim.zs[order][s:e]} μm\nindex\n{order[s:e]}')

4. Design secondary lens
    simulate central light from primary lens, further to secondary lens location
    to obtain secondary lens design

In [ ]:
# this run very long, but is more accurate than ASM

# path = f'{c.path.E}'
# z=1000
# λ = c.λ/n_G
# RS.RS_batch( (E_center_0*primary_lens, c.A.xs, c.A.ys),\
#             (c.sim.xs, c.sim.ys, z, c.λ),\
#                 (16, path))
# time = datetime.now()
# print(f'last run: {time.isoformat(timespec="minutes")}')

In [ ]:
# E_center_1000 = np.load(f'{path}/Z=1000.npy')
# time = datetime.now()
# print(f'last run: {time.isoformat(timespec="minutes")}')

In [ ]:
c.sim.z_start = 0
c.sim.z_end = 1100
c.sim.Nz = 11
c.sim.zs = np.linspace(c.sim.z_start, c.sim.z_end, c.sim.Nz, endpoint=False)
c.sim.Δz = c.sim.zs[1] - c.sim.zs[0]
pad_value = 1200
mul = 2*pad_value//len(E_center_0)+1
ASM.ASM_3D_batch_E_Multi_Process(A_xy=it.img_pad(E_center_0*primary_lens, (pad_value,pad_value,pad_value,pad_value), pad_val=0), Lx = c.A.Lx*mul, Ly=c.A.Ly*mul, zs=c.sim.zs,\
                                  λ=c.λ/n_G, path=f'{c.path.E}/tmp',\
                                  batch_size=4, xy_range_idx=(pad_value, -pad_value, pad_value, -pad_value), num_process=6)
time = datetime.now()
print(f'last run: {time.isoformat(timespec="minutes")}')

In [ ]:
# extract results
E_center_0_1100_11 = ASM.batch_E_extract(c.A.Nx, c.A.Ny, len(c.sim.zs), f'{c.path.E}/tmp', batch_size=4)
time = datetime.now()
print(f'last run: {time.isoformat(timespec="minutes")}')
E_center_1000 = E_center_0_1100_11[:, :, 10]

In [ ]:
E_target_1000 = 1
E_relative_1000 = E_center_1000 / E_target_1000
θ_aperture = np.angle(E_relative_1000)
cos_θ_aperture_θ0s = np.cos(θ_aperture[..., None]-θ0s) # Nx, Ny, Nθs
# print(cos_θ_aperture_θ0s.shape)
edge = it.block_edge_mask(c.A.Nx, c.A.Ny, int(300/2/c.A.Lx*c.A.Nx)) # Nx, Ny # circle mask of Nx x Ny pixel with pixel radius
# apply constraint of 100μm diameter
aperture_design = (cos_θ_aperture_θ0s > 0) * ( edge[:, :, None] == True ) # set edge to block
# find best possible match over range of reference phase
#                  phase_similarity (Nx, Ny, Nθs)
cosine_similarity = (cos_θ_aperture_θ0s * aperture_design * np.abs(E_center_1000)[:, :, None]).sum(axis=(0, 1)) # Nθs (range of reference phase)
idx_max = cosine_similarity.argmax()
idx_min = cosine_similarity.argmin()
print(f'min similarity phase / max similarity phase\n{cosine_similarity[idx_min] / cosine_similarity[idx_max]}')
import gc
del cos_θ_aperture_θ0s
secondary_lens = aperture_design[:, :, idx_max]
del aperture_design
gc.collect()

In [ ]:
plt.clf()
plt.imshow(secondary_lens.T)
plt.show()
name = 'LED_-220μm_2_-22μm_λ620nm_300_pad2700_secondary_ASM'
it.array_2_img(secondary_lens, f'{c.path.results}/{name}.bmp')

5. Obtain energy-angle distribution of test light

In [31]:
intensity_dist_secondary_lens, kθ, xθ, yθ = ASM.Angular_intensity_dist(E_test_1000*secondary_lens, c.sim.Lx, c.sim.Ly, λ=c.λ)
intensity_dist, kθ, xθ, yθ = ASM.Angular_intensity_dist(E_test_1000, c.sim.Lx, c.sim.Ly, λ=c.λ)
f_range = 50; Nx, Ny = intensity_dist.shape

x_angles = np.fft.fftshift(xθ)[Nx // 2 - f_range: Nx // 2 + f_range, Nx // 2 - f_range: Nx // 2 + f_range]
y_angles = np.fft.fftshift(yθ)[Ny // 2 - f_range: Ny // 2 + f_range, Ny // 2 - f_range: Ny // 2 + f_range]
x_angle_ticks = np.linspace(0, len(x_angles), len(x_angles)//20, endpoint=False, dtype=int)
y_angle_ticks = np.linspace(0, len(y_angles), len(y_angles)//20, endpoint=False, dtype=int)

plt.clf()
plt.title(f"intensity vs angle (|FFT_A|^2)")
plt.imshow(np.fft.fftshift(intensity_dist)[Nx // 2 - f_range: Nx // 2 + f_range, Ny // 2 - f_range: Ny // 2 + f_range].T); plt.gca().invert_yaxis()
plt.xticks(x_angle_ticks, np.round(x_angles[x_angle_ticks, 0]).astype(int))
plt.yticks(y_angle_ticks, np.round(y_angles[0, y_angle_ticks]).astype(int))
plt.show()

NameError: name 'E_test_1000' is not defined

In [ ]:
plt.clf()
total_intensity = intensity_dist.sum()
cone_angle = np.array([1, 2, 3, 4, 5])
cone_intensity = (intensity_dist[:, :, None] * ((kθ.real)[:, :, None] < cone_angle)).sum(axis=(0,1))
print(f'number of sample point: {((kθ.real)[:, :, None] < cone_angle).sum(axis=(0,1))}')



plt.plot(cone_angle, cone_intensity / total_intensity)
plt.xticks()
plt.title('total intensity ratio within forward cone\nvs cone angle (degree)')
plt.show()

In [ ]:
print(intensity_dist.sum())
print(np.fft.fftshift(intensity_dist)[Nx // 2 - f_range: Nx // 2 + f_range, Nx // 2 - f_range: Nx // 2 + f_range].sum()/intensity_dist.sum())